# Traffic Light DQN Training - Google Colab

This notebook trains a Deep Q-Network agent to optimize traffic light control at a 4-way intersection.

**Time required:** ~30-60 minutes for 100 episodes on Colab GPU

## 1. Setup and Install Dependencies

In [ ]:
# Install required packages
!pip install torch numpy matplotlib pygame -q

import sys
print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")

## 2. Import Required Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
from collections import deque, namedtuple
import matplotlib.pyplot as plt
import time
import os
from datetime import datetime

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Check GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 3. Clone or Mount Project Files

**Choose one option below:**

### Option A: Clone from GitHub (if your project is on GitHub)

In [ ]:
# Clone repository (replace with your repo URL)
# !git clone https://github.com/yourusername/traffic-light-dqn.git
# %cd traffic-light-dqn

print("Skipped - Use Option B instead")

### Option B: Mount Google Drive

In [ ]:
from google.colab import drive

# Mount your Google Drive
drive.mount('/content/drive')

# Update the path below to match your project location
project_path = '/content/drive/MyDrive/traffic'  # Adjust this path
os.chdir(project_path)
print(f"Working directory: {os.getcwd()}")
print(f"Files: {os.listdir('.')}")

## 4. Define Simulation Configuration

In [ ]:
# Training Configuration
class TrainingConfig:
    """Training parameters for DQN"""
    
    # Training parameters
    TRAINING_EPISODES = 100  # Increase for better results (100-500 recommended)
    STEPS_PER_EPISODE = 500  # Steps per episode
    DQN_DECISION_INTERVAL = 8  # Seconds between decisions
    
    # DQN Network Parameters
    DQN_PARAMS = {
        'learning_rate': 0.0001,
        'gamma': 0.99,
        'epsilon_start': 1.0,
        'epsilon_end': 0.01,
        'epsilon_decay': 0.9997,
        'batch_size': 32,
        'memory_size': 10000,
        'target_update': 100
    }
    
    # Reward Structure
    DQN_REWARDS = {
        'vehicle_crossed': 10.0,
        'waiting_reduction': 2.0,
        'waiting_increase_penalty': -1.0,
        'traffic_balance_bonus': 2.0,
    }

config = TrainingConfig()
print("Training Configuration:")
print(f"  Episodes: {config.TRAINING_EPISODES}")
print(f"  Learning Rate: {config.DQN_PARAMS['learning_rate']}")
print(f"  Gamma: {config.DQN_PARAMS['gamma']}")

## 5. Define DQN Neural Network and Agent

In [ ]:
# Experience Replay
Experience = namedtuple('Experience', ['state', 'action', 'reward', 'next_state', 'done'])

class ReplayMemory:
    """Experience Replay Memory for DQN"""
    
    def __init__(self, capacity):
        self.memory = deque(maxlen=capacity)
    
    def push(self, state, action, reward, next_state, done):
        self.memory.append(Experience(state, action, reward, next_state, done))
    
    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)
    
    def __len__(self):
        return len(self.memory)

print("Replay Memory defined")

In [ ]:
# DQN Network Architecture
class DQNNetwork(nn.Module):
    """Deep Q-Network for traffic control"""
    
    def __init__(self, state_size, action_size, hidden_size=128):
        super(DQNNetwork, self).__init__()
        
        self.fc1 = nn.Linear(state_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, hidden_size)
        self.fc4 = nn.Linear(hidden_size, action_size)
        
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.fc4(x)
        return x

print("DQN Network defined")

In [ ]:
# DQN Traffic Agent
class DQNTrafficAgent:
    """DQN Agent for Traffic Light Control"""
    
    def __init__(self, config, rewards, device):
        self.config = config
        self.rewards = rewards
        self.device = device
        
        # State and action sizes
        self.state_size = 17  # Calculated from signal state, traffic state, etc.
        self.action_size = 5  # 4 signals + hold current
        
        # Networks
        self.policy_net = DQNNetwork(self.state_size, self.action_size, hidden_size=128).to(device)
        self.target_net = DQNNetwork(self.state_size, self.action_size, hidden_size=128).to(device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()
        
        # Optimizer and loss
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=config['learning_rate'])
        self.criterion = nn.MSELoss()
        
        # Memory
        self.memory = ReplayMemory(config['memory_size'])
        
        # Training parameters
        self.gamma = config['gamma']
        self.epsilon = config['epsilon_start']
        self.epsilon_end = config['epsilon_end']
        self.epsilon_decay = config['epsilon_decay']
        self.batch_size = config['batch_size']
        self.target_update = config['target_update']
        
        # Stats
        self.steps_done = 0
        self.episodes_done = 0
        self.total_rewards = []
        self.losses = []
    
    def select_action(self, state, explore=True):
        """Select action using epsilon-greedy policy"""
        if explore and random.random() < self.epsilon:
            return random.randrange(self.action_size)
        
        with torch.no_grad():
            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(self.device)
            q_values = self.policy_net(state_tensor)
            return q_values.max(1)[1].item()
    
    def store_experience(self, state, action, reward, next_state, done):
        """Store experience in replay memory"""
        self.memory.push(state, action, reward, next_state, done)
    
    def train(self):
        """Train the DQN network"""
        if len(self.memory) < self.batch_size:
            return None
        
        # Sample batch
        experiences = self.memory.sample(self.batch_size)
        batch = Experience(*zip(*experiences))
        
        # Convert to tensors
        state_batch = torch.FloatTensor(np.array(batch.state)).to(self.device)
        action_batch = torch.LongTensor(batch.action).to(self.device)
        reward_batch = torch.FloatTensor(batch.reward).to(self.device)
        next_state_batch = torch.FloatTensor(np.array(batch.next_state)).to(self.device)
        done_batch = torch.FloatTensor(batch.done).to(self.device)
        
        # Compute Q values
        current_q_values = self.policy_net(state_batch).gather(1, action_batch.unsqueeze(1))
        
        # Compute target Q values
        with torch.no_grad():
            next_q_values = self.target_net(next_state_batch).max(1)[0]
            target_q_values = reward_batch + self.gamma * next_q_values * (1 - done_batch)
        
        # Compute loss and update
        loss = self.criterion(current_q_values.squeeze(), target_q_values)
        
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        # Update target network
        self.steps_done += 1
        if self.steps_done % self.target_update == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())
        
        return loss.item()
    
    def update_epsilon(self):
        """Decay epsilon for exploration"""
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)
    
    def save_model(self, filepath):
        """Save model weights"""
        torch.save({
            'policy_net': self.policy_net.state_dict(),
            'target_net': self.target_net.state_dict(),
            'episodes_done': self.episodes_done,
            'steps_done': self.steps_done,
            'epsilon': self.epsilon,
        }, filepath)
        print(f"Model saved to {filepath}")
    
    def load_model(self, filepath):
        """Load model weights"""
        checkpoint = torch.load(filepath)
        self.policy_net.load_state_dict(checkpoint['policy_net'])
        self.target_net.load_state_dict(checkpoint['target_net'])
        self.episodes_done = checkpoint['episodes_done']
        self.steps_done = checkpoint['steps_done']
        self.epsilon = checkpoint['epsilon']
        print(f"Model loaded from {filepath}")

print("DQN Agent defined")

## 6. Define Headless Traffic Simulation

In [ ]:
# Simple headless traffic simulation for training
class SimpleTrafficSimulation:
    """Simplified traffic simulation without pygame (for Colab)"""
    
    def __init__(self):
        self.vehicles = {d: [] for d in ['right', 'down', 'left', 'up']}
        self.signal_timers = [0, 0, 0, 0]  # Green time remaining for each signal
        self.current_green_signal = 0
        self.signal_durations = [10, 10, 10, 10]
        self.steps = 0
    
    def step(self, action=None):
        """Step simulation by one time unit"""
        self.steps += 1
        
        # Generate random vehicles
        for direction in ['right', 'down', 'left', 'up']:
            if random.random() < 0.3:  # 30% chance per step
                self.vehicles[direction].append({'wait_time': 0})
        
        # Increment wait times
        for direction in ['right', 'down', 'left', 'up']:
            for vehicle in self.vehicles[direction]:
                vehicle['wait_time'] += 1
        
        # Handle vehicle movement through green light
        crossed = 0
        if self.signal_timers[self.current_green_signal] > 0:
            direction = ['right', 'down', 'left', 'up'][self.current_green_signal]
            while self.vehicles[direction] and self.signal_timers[self.current_green_signal] > 0:
                vehicle = self.vehicles[direction].pop(0)
                crossed += 1
                self.signal_timers[self.current_green_signal] -= max(1, vehicle['wait_time'] // 10)
        
        # Update signal timers
        for i in range(4):
            if i == self.current_green_signal and self.signal_timers[i] > 0:
                self.signal_timers[i] -= 1
        
        # Switch signal if timer expired
        if self.signal_timers[self.current_green_signal] <= 0:
            self.current_green_signal = (self.current_green_signal + 1) % 4
            self.signal_timers[self.current_green_signal] = self.signal_durations[self.current_green_signal]
        
        return crossed, self.get_state()
    
    def get_state(self):
        """Get current simulation state"""
        directions = ['right', 'down', 'left', 'up']
        state = np.zeros(17, dtype=np.float32)
        
        # Signal timers (4 values)
        for i in range(4):
            state[i] = min(self.signal_timers[i], 60) / 60.0
        
        # Waiting vehicles (4 values)
        for i, direction in enumerate(directions):
            state[4 + i] = min(len(self.vehicles[direction]), 30) / 30.0
        
        # Traffic levels (4 values) - based on waiting vehicles
        for i, direction in enumerate(directions):
            traffic_level = min(len(self.vehicles[direction]) * 1.5, 10)
            state[8 + i] = traffic_level / 10.0
        
        # Current green signal (4 one-hot)
        state[12 + self.current_green_signal] = 1.0
        
        # Yellow phase (1 value)
        state[16] = 0.0
        
        return state
    
    def execute_action(self, action):
        """Execute an action (switch signal)"""
        if action < 4:
            self.current_green_signal = action
            self.signal_timers[action] = self.signal_durations[action]

print("Simple Traffic Simulation defined")

## 7. Training Loop

In [ ]:
# Initialize simulation and agent
simulation = SimpleTrafficSimulation()
agent = DQNTrafficAgent(config.DQN_PARAMS, config.DQN_REWARDS, device)

print(f"\n{'='*70}")
print(" " * 15 + "DQN TRAFFIC LIGHT TRAINING - GOOGLE COLAB")
print(f"{'='*70}")
print(f"Device: {device}")
print(f"Episodes: {config.TRAINING_EPISODES}")
print(f"Steps per episode: {config.STEPS_PER_EPISODE}")
print(f"State size: {agent.state_size}")
print(f"Action size: {agent.action_size}")
print(f"{'='*70}\n")

In [ ]:
# Training loop
start_time = time.time()
episode_rewards = []
episode_losses = []
episode_vehicles_crossed = []

for episode in range(config.TRAINING_EPISODES):
    # Reset episode
    simulation = SimpleTrafficSimulation()
    state = simulation.get_state()
    episode_reward = 0
    episode_loss = []
    episode_crossed = 0
    
    # Episode loop
    for step in range(config.STEPS_PER_EPISODE):
        # Select and execute action
        action = agent.select_action(state, explore=True)
        simulation.execute_action(action)
        
        # Step simulation
        crossed, next_state = simulation.step(action)
        episode_crossed += crossed
        
        # Calculate reward
        reward = 0
        if crossed > 0:
            reward += crossed * config.DQN_REWARDS['vehicle_crossed']
        
        episode_reward += reward
        
        # Store experience
        done = False
        agent.store_experience(state, action, reward, next_state, done)
        
        # Train
        loss = agent.train()
        if loss is not None:
            episode_loss.append(loss)
        
        state = next_state
    
    # End of episode
    agent.episodes_done += 1
    agent.update_epsilon()
    agent.total_rewards.append(episode_reward)
    
    episode_rewards.append(episode_reward)
    if episode_loss:
        episode_losses.append(np.mean(episode_loss))
    else:
        episode_losses.append(0)
    episode_vehicles_crossed.append(episode_crossed)
    
    # Progress report
    if (episode + 1) % max(1, config.TRAINING_EPISODES // 10) == 0:
        elapsed = time.time() - start_time
        print(f"Episode {episode+1}/{config.TRAINING_EPISODES} | "
              f"Reward: {episode_reward:.1f} | "
              f"Loss: {episode_losses[-1]:.4f} | "
              f"Epsilon: {agent.epsilon:.4f} | "
              f"Time: {elapsed/60:.1f}m")

training_time = time.time() - start_time
print(f"\nTraining completed in {training_time/60:.1f} minutes")

## 8. Save Trained Model

In [ ]:
# Create models directory if it doesn't exist
os.makedirs('models', exist_ok=True)

# Save the trained model
model_path = 'models/dqn_traffic_colab.pth'
agent.save_model(model_path)

# Also save a backup
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
backup_path = f'models/dqn_traffic_colab_{timestamp}.pth'
agent.save_model(backup_path)

print(f"\nModel saved successfully!")
print(f"  Main: {model_path}")
print(f"  Backup: {backup_path}")

## 9. Visualize Training Performance

In [ ]:
# Create performance plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Episode Rewards
axes[0, 0].plot(episode_rewards, linewidth=2, color='blue')
axes[0, 0].set_title('Episode Rewards Over Time', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Episode')
axes[0, 0].set_ylabel('Total Reward')
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Training Loss
axes[0, 1].plot(episode_losses, linewidth=2, color='red')
axes[0, 1].set_title('Training Loss Over Time', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Episode')
axes[0, 1].set_ylabel('Average Loss')
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Vehicles Crossed
axes[1, 0].bar(range(len(episode_vehicles_crossed)), episode_vehicles_crossed, color='green', alpha=0.7)
axes[1, 0].set_title('Vehicles Crossed per Episode', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Episode')
axes[1, 0].set_ylabel('Count')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Plot 4: Epsilon Decay
epsilon_values = [config.DQN_PARAMS['epsilon_start'] * (config.DQN_PARAMS['epsilon_decay'] ** i) 
                   for i in range(config.TRAINING_EPISODES)]
axes[1, 1].plot(epsilon_values, linewidth=2, color='orange')
axes[1, 1].set_title('Epsilon Exploration Rate Decay', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Episode')
axes[1, 1].set_ylabel('Epsilon')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_performance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTraining performance plots saved as 'training_performance.png'")

## 10. Training Summary

In [ ]:
# Training summary statistics
print(f"\n{'='*70}")
print(" " * 20 + "TRAINING SUMMARY")
print(f"{'='*70}")
print(f"Total Episodes: {config.TRAINING_EPISODES}")
print(f"Total Training Time: {training_time/60:.2f} minutes")
print(f"Average Time per Episode: {training_time/config.TRAINING_EPISODES:.2f} seconds")
print(f"\nReward Statistics:")
print(f"  Average Reward: {np.mean(episode_rewards):.2f}")
print(f"  Max Reward: {np.max(episode_rewards):.2f}")
print(f"  Min Reward: {np.min(episode_rewards):.2f}")
print(f"  Std Dev: {np.std(episode_rewards):.2f}")
print(f"\nLoss Statistics:")
print(f"  Average Loss: {np.mean(episode_losses):.4f}")
print(f"  Final Loss: {episode_losses[-1]:.4f}")
print(f"\nVehicles Crossed:")
print(f"  Total: {sum(episode_vehicles_crossed)}")
print(f"  Average per Episode: {np.mean(episode_vehicles_crossed):.0f}")
print(f"\nAgent Statistics:")
print(f"  Final Epsilon: {agent.epsilon:.4f}")
print(f"  Total Steps: {agent.steps_done}")
print(f"  Replay Memory Size: {len(agent.memory)}")
print(f"\nModel Saved:")
print(f"  {model_path}")
print(f"  Size: {os.path.getsize(model_path) / 1024:.2f} KB")
print(f"{'='*70}")

## 11. Download Model (Optional)

If you're using Google Colab, run this cell to download the trained model:

In [ ]:
try:
    from google.colab import files
    print("Downloading model files...")
    files.download(model_path)
    print("Download complete!")
except ImportError:
    print("Not running in Google Colab. Model is already in your project directory.")